Importation des bibliothèque

In [ ]:
import numpy as np
from scipy.stats import genpareto, poisson
from scipy.stats import percentileofscore

Mise en As-If

In [ ]:
# --- CONFIGURATION DES ASSURÉS ---
# 'type_franchise' : 'proportion' (ex: 5% du brut) ou 'fixe' (montant en euros)
dict_assures = {
    'Assuré_A': {
        'proportion': 0.11, 
        'type_franchise': 'fixe', 
        'valeur': 750_000_000,    # 5%
        'min_f':  0, # Minimum de 1M€
        'retention_cedante': 500_000_000
    },
    'Assuré_B': {
        'proportion': 0.27, 
        'type_franchise': 'fixe', 
        'valeur': 1_500_000_000, # 15M€ fixe peu importe le sinistre
        'min_f': 0, # Pas de minimum nécessaire ici
        'retention_cedante': 500_000_000
    },
    'Assuré_C': {
        'proportion': 0.11, 
        'type_franchise': 'proportion', 
        'valeur': 0.1,    # 8%
        'min_f': 150_000_000, # Minimum de 5M€
        'retention_cedante': 500_000_000
    },
    'Autres': {
        'proportion': 0.51, 
        'type_franchise': 'proportion', 
        'valeur': 0.05, 
        'min_f': 0,
        'retention_cedante': 300_000_000
    }
}

TARIFICATION

1) Par simulation 

1.1) 1st Layer

In [ ]:
# --- 1. PARAMÈTRES DE L'ÉTUDE (TRANCHE 1) ---
xi = 0.4003      
sigma = 1_748_298_084.89   
u = 600_000_000          
lam = 4.6               
priority = 1_000_000_000       # Priorité conservée par la CCR sur sa part nette
capacity = 2_000_000_000     # Capacité fournie par le rétrocessionnaire
n_annees = 100000
n_reinstatements = 3         
EPI = 5_000_000_000          # Assiette de prime (Primes encaissées par la CCR)

# --- PARAMÈTRE DE PART DE LA CCR ---
part_ccr_dans_sinistre = 0.60  # La CCR détient 50% de chaque sinistre brut

# --- 2. GÉNÉRATION DE L'ÉCHANTILLON DE SÉVÉRITÉ (GPD) ---
np.random.seed(42)
echantillons_exces = genpareto.rvs(xi, loc=0, scale=sigma, size=10000)
echantillons_nouveau = echantillons_exces + u

# --- 3. FONCTION DE SIMULATION ADAPTÉE (RÉTROCESSION) ---
def simuler_tarification_retro(echantillons_gpd, lam, priority, capacity, n_reinst, n_sim, part_ccr):
    charges_annuelles = []
    recons_payees_annuelles = [] 

    noms_assures = list(dict_assures.keys())
    poids = [dict_assures[a]['proportion'] for a in noms_assures]

    for _ in range(n_sim):
        n_sin = poisson.rvs(lam)
        charge_annee = 0
        # La capacité limite annuelle du rétrocessionnaire
        capacite_limite_annuelle = capacity * (1 + n_reinst)
        recons_cette_annee = 0
        
        if n_sin > 0:
            sinistres_bruts = np.random.choice(echantillons_gpd, size=n_sin)
            assures_concernes = np.random.choice(noms_assures, size=n_sin, p=poids)
            
            for i in range(n_sin):
                s_brut = sinistres_bruts[i]
                assure = assures_concernes[i]
                conf = dict_assures[assure]
                
                # --- A. DÉDUCTIONS SUR LE BRUT (100%) ---
                if conf['type_franchise'] == 'proportion':
                    franchise = max(conf['min_f'], conf['valeur'] * s_brut)
                else:
                    franchise = conf['valeur']
                
                s_net_f_100 = max(0, s_brut - franchise)
                s_net_cedante_100 = max(0, s_net_f_100 - conf['retention_cedante'])
                
                # --- B. APPLICATION DE LA PART CCR ---
                # Ce qui remonte réellement à la CCR
                s_part_ccr = s_net_cedante_100 * part_ccr
                
                # --- C. IMPACT SUR LE TRAITÉ DE RÉTRO ---
                # Le rétrocessionnaire intervient au-delà de la priorité de la CCR
                impact = max(0, min(s_part_ccr - priority, capacity))
                
                if impact > 0:
                    paiement = min(impact, capacite_limite_annuelle - charge_annee)
                    charge_annee += paiement
                    recons_cette_annee += (paiement / capacity)

        charges_annuelles.append(charge_annee)
        recons_payees_annuelles.append(recons_cette_annee)

    return np.array(charges_annuelles), np.array(recons_payees_annuelles)

# --- 4. EXÉCUTION ET CALCUL ---
charges, recons = simuler_tarification_retro(
    echantillons_nouveau, lam, priority, capacity, n_reinstatements, n_annees, part_ccr_dans_sinistre
)

E_charge = np.mean(charges)
E_recons = np.mean(recons)

# Prime Pure Nette (MDP)
prime_pure_nette = E_charge / (1 + E_recons)
taux_sur_epi = (prime_pure_nette / EPI) * 100

# --- 5. RÉSULTATS ---
print(f"--- SYNTHÈSE RÉTROCESSION (TRANCHE 1) ---")
print(f"Part CCR sur sinistres bruts : {part_ccr_dans_sinistre*100}%")
print(f"Assiette de prime (EPI)      : {EPI:,.0f} DA")
print("-" * 45)
print(f"Charge annuelle moyenne (E[Loss])   : {E_charge:,.2f} DA")
print(f"Espérance de reconstitution payée   : {E_recons:.4f}")
print("-" * 45)
print(f"PRIME PURE DE RÉTRO (MDP)           : {prime_pure_nette:,.2f} DA")
print(f"TAUX À APPLIQUER À L'EPI            : {taux_sur_epi:.4f} %")
print("-" * 45)

1.2) 2nd Layer

In [ ]:
# --- 1. PARAMÈTRES DE L'ÉTUDE (TRANCHE 2) ---
xi = 0.4003      
sigma = 1_748_298_084.89   
u = 600_000_000          
lam = 2               
priority = 3_000_000_000     # Priorité conservée par la CCR sur sa part nette
capacity = 3_000_000_000     # Capacité fournie par le rétrocessionnaire
n_annees = 100000
n_reinstatements = 2         
EPI = 5_000_000_000          # Assiette de prime (Primes encaissées par la CCR)

# --- PARAMÈTRE DE PART DE LA CCR ---
part_ccr_dans_sinistre = 0.60  # La CCR détient 50% de chaque sinistre brut

# --- 2. GÉNÉRATION DE L'ÉCHANTILLON DE SÉVÉRITÉ (GPD) ---
np.random.seed(42)
echantillons_exces = genpareto.rvs(xi, loc=0, scale=sigma, size=10000)
echantillons_nouveau = echantillons_exces + u

# --- 3. FONCTION DE SIMULATION ADAPTÉE (RÉTROCESSION) ---
def simuler_tarification_retro(echantillons_gpd, lam, priority, capacity, n_reinst, n_sim, part_ccr):
    charges_annuelles = []
    recons_payees_annuelles = [] 

    noms_assures = list(dict_assures.keys())
    poids = [dict_assures[a]['proportion'] for a in noms_assures]

    for _ in range(n_sim):
        n_sin = poisson.rvs(lam)
        charge_annee = 0
        # Capacité limite annuelle (Capacité x (1 + nb recons))
        capacite_limite_annuelle = capacity * (1 + n_reinst)
        recons_cette_annee = 0
        
        if n_sin > 0:
            sinistres_bruts = np.random.choice(echantillons_gpd, size=n_sin)
            assures_concernes = np.random.choice(noms_assures, size=n_sin, p=poids)
            
            for i in range(n_sin):
                s_brut = sinistres_bruts[i]
                assure = assures_concernes[i]
                conf = dict_assures[assure]
                
                # --- A. DÉDUCTIONS SUR LE BRUT (100%) ---
                if conf['type_franchise'] == 'proportion':
                    franchise = max(conf['min_f'], conf['valeur'] * s_brut)
                else:
                    franchise = conf['valeur']
                
                s_net_f_100 = max(0, s_brut - franchise)
                s_net_cedante_100 = max(0, s_net_f_100 - conf['retention_cedante'])
                
                # --- B. APPLICATION DE LA PART CCR ---
                # La CCR ne récupère sa part que sur le sinistre déjà net de déductions
                s_part_ccr = s_net_cedante_100 * part_ccr
                
                # --- C. IMPACT SUR LE TRAITÉ DE RÉTRO ---
                # Le rétrocessionnaire intervient au-delà de la priorité de la CCR
                impact = max(0, min(s_part_ccr - priority, capacity))
                
                if impact > 0:
                    paiement = min(impact, capacite_limite_annuelle - charge_annee)
                    charge_annee += paiement
                    # Reconstitution calculée sur la base de la capacité faciale
                    recons_cette_annee += (paiement / capacity)

        charges_annuelles.append(charge_annee)
        recons_payees_annuelles.append(recons_cette_annee)

    return np.array(charges_annuelles), np.array(recons_payees_annuelles)

# --- 4. EXÉCUTION ET CALCUL DES TAUX ---
charges, recons = simuler_tarification_retro(
    echantillons_nouveau, lam, priority, capacity, n_reinstatements, n_annees, part_ccr_dans_sinistre
)

E_charge = np.mean(charges)
E_recons = np.mean(recons)

# Prime Pure Nette (MDP)
prime_pure_nette = E_charge / (1 + E_recons)
taux_sur_epi = (prime_pure_nette / EPI) * 100

# --- 5. RÉSULTATS ---
print(f"--- SYNTHÈSE RÉTROCESSION (TRANCHE 2 : 3 Md XS 3 Md) ---")
print(f"Part CCR sur sinistres bruts : {part_ccr_dans_sinistre*100}%")
print(f"Assiette de prime (EPI)      : {EPI:,.0f} DA")
print("-" * 55)
print(f"Charge annuelle moyenne (E[Loss])   : {E_charge:,.2f} DA")
print(f"Espérance de reconstitution payée   : {E_recons:.4f}")
print("-" * 55)
print(f"PRIME PURE DE RÉTRO (MDP)           : {prime_pure_nette:,.2f} DA")
print(f"TAUX À APPLIQUER À L'EPI            : {taux_sur_epi:.4f} %")
print("-" * 55)

1.3) 3rd Layer

In [ ]:
import numpy as np
from scipy.stats import genpareto, poisson

# --- 1. PARAMÈTRES DE L'ÉTUDE (TRANCHE 3) ---
xi = 0.4003      
sigma = 1_748_298_084.89   
u = 600_000_000         
lam = 1               
priority = 6_000_000_000     # Priorité conservée par la CCR sur sa part nette
capacity = 6_000_000_000    # Capacité fournie par le rétrocessionnaire
n_annees = 100000
n_reinstatements = 1         
EPI = 5_000_000_000          # Assiette de prime (Primes encaissées par la CCR)

# --- PARAMÈTRE DE PART DE LA CCR ---
part_ccr_dans_sinistre = 0.60  # La CCR détient 50% de chaque sinistre brut

# --- 2. GÉNÉRATION DE L'ÉCHANTILLON DE SÉVÉRITÉ (GPD) ---
np.random.seed(42)
echantillons_exces = genpareto.rvs(xi, loc=0, scale=sigma, size=10000)
echantillons_nouveau = echantillons_exces + u

# --- 3. FONCTION DE SIMULATION ADAPTÉE (RÉTROCESSION) ---
def simuler_tarification_retro(echantillons_gpd, lam, priority, capacity, n_reinst, n_sim, part_ccr):
    charges_annuelles = []
    recons_payees_annuelles = [] 

    # Préparation pour le tirage des assurés
    noms_assures = list(dict_assures.keys())
    poids = [dict_assures[a]['proportion'] for a in noms_assures]

    for _ in range(n_sim):
        n_sin = poisson.rvs(lam)
        charge_annee = 0
        # Capacité limite annuelle incluant les reconstitutions
        capacite_limite_annuelle = capacity * (1 + n_reinst)
        recons_cette_annee = 0
        
        if n_sin > 0:
            sinistres_bruts = np.random.choice(echantillons_gpd, size=n_sin)
            assures_concernes = np.random.choice(noms_assures, size=n_sin, p=poids)
            
            for i in range(n_sin):
                s_brut = sinistres_bruts[i]
                assure = assures_concernes[i]
                conf = dict_assures[assure]
                
                # --- A. DÉDUCTIONS SUR LE BRUT (100%) ---
                if conf['type_franchise'] == 'proportion':
                    franchise = max(conf['min_f'], conf['valeur'] * s_brut)
                else:
                    franchise = conf['valeur']
                
                s_net_f_100 = max(0, s_brut - franchise)
                s_net_cedante_100 = max(0, s_net_f_100 - conf['retention_cedante'])
                
                # --- B. APPLICATION DE LA PART CCR ---
                # La CCR ne récupère sa quote-part que sur le sinistre déjà net de franchise/rétention
                s_part_ccr = s_net_cedante_100 * part_ccr
                
                # --- C. IMPACT SUR LE TRAITÉ DE RÉTRO ---
                # Le rétrocessionnaire paie l'excédent au-delà de la priorité de la CCR
                impact = max(0, min(s_part_ccr - priority, capacity))
                
                if impact > 0:
                    paiement = min(impact, capacite_limite_annuelle - charge_annee)
                    charge_annee += paiement
                    # Reconstitution calculée sur la base de la capacité faciale de la tranche
                    recons_cette_annee += (paiement / capacity)

        charges_annuelles.append(charge_annee)
        recons_payees_annuelles.append(recons_cette_annee)

    return np.array(charges_annuelles), np.array(recons_payees_annuelles)

# --- 4. EXÉCUTION ET CALCUL DES TAUX ---
charges, recons = simuler_tarification_retro(
    echantillons_nouveau, lam, priority, capacity, n_reinstatements, n_annees, part_ccr_dans_sinistre
)

E_charge = np.mean(charges)
E_recons = np.mean(recons)

# Prime Pure Nette (MDP) tenant compte des reconstitutions
prime_pure_nette = E_charge / (1 + E_recons)
taux_sur_epi = (prime_pure_nette / EPI) * 100

# --- 5. RÉSULTATS ---
print(f"--- SYNTHÈSE RÉTROCESSION (TRANCHE 3 : 11 Md XS 6 Md) ---")
print(f"Part CCR sur sinistres bruts : {part_ccr_dans_sinistre*100}%")
print(f"Assiette de prime (EPI)      : {EPI:,.0f} DA")
print("-" * 55)
print(f"Charge annuelle moyenne (E[Loss])   : {E_charge:,.2f} DA")
print(f"Espérance de reconstitution payée   : {E_recons:.4f}")
print("-" * 55)
print(f"PRIME PURE DE RÉTRO (MDP)           : {prime_pure_nette:,.2f} DA")
print(f"TAUX À APPLIQUER À L'EPI            : {taux_sur_epi:.4f} %")
print("-" * 55)

TARIFICATION

2. Par le modèle collectif

In [ ]:
import numpy as np
from scipy.stats import genpareto

# --- 1. PARAMÈTRES DE LA LOI GPD ET DE LA FRÉQUENCE ---
xi = 0.4003      
sigma = 1_748_298_084.89   
u = 600_000_000           
lam = 4.6                
part_ccr = 0.60

# Dictionnaire simulant le comportement moyen du marché (Franchises / Rétentions cédantes)
# Nécessaire pour passer du brut GPD au "Net de premier niveau" entrant à la CCR
dict_assures = {
    'Assure_Moyen': {'proportion': 1.0, 'type_franchise': 'fixe', 'valeur': 50_000_000, 'retention_cedante': 100_000_000}
}

# --- 2. FONCTION DE CALCUL ANALYTIQUE DU LEV (GPD) ---
def limited_expected_value_gpd(K, xi, sigma, u):
    """Calcule E[min(X, K)] pour une variable X suivant une GPD de seuil u"""
    if K <= u:
        return K
    
    # Formule analytique de l'espérance limitée de la GPD
    puissance = 1 - (1 / xi)
    terme_gpd = (sigma / (1 - xi)) * (1 - (1 + (xi * (K - u)) / sigma) ** puissance)
    return u + terme_gpd

# --- 3. DÉFINITION DES TROIS TRANCHES D'ORIGINE (À LA CHARGE DE LA CCR) ---
# Tranche 1 : 2 Mds XS 1 Md
# Tranche 2 : 3 Mds XS 3 Mds (Prend au-dessus de 1Md + 2Mds = 3Mds)
# Tranche 3 : 6 Mds XS 6 Mds (Prend au-dessus de 3Mds + 3Mds = 6Mds)
tranches_config = [
    {"nom": "Tranche 1 (2Mds XS 1Md)", "priorite": 1_000_000_000, "portee": 2_000_000_000},
    {"nom": "Tranche 2 (3Mds XS 3Mds)", "priorite": 3_000_000_000, "portee": 3_000_000_000},
    {"nom": "Tranche 3 (6Mds XS 6Mds)", "priorite": 6_000_000_000, "portee": 6_000_000_000}
]

print(f"--- TARIFICATION ANALYTIQUE DES TRAITÉS (PRIME PURE THÉORIQUE) ---")
print("-" * 75)

# --- 4. CALCUL ET INTÉGRATION DES CONTRAINTES ---
# Pour une précision analytique pure sur le portefeuille, on évalue l'impact moyen
conf = dict_assures['Assure_Moyen']

for t in tranches_config:
    L = t["priorite"]
    M = t["portee"]
    L_plus_M = L + M
    
    # Étape A : Pour atteindre les seuils nets L et L+M au niveau de la CCR (60%),
    # il faut retrouver à quels montants bruts (100% GPD) cela correspond à l'entrée.
    # Formule inverse de : Part_CCR * Max(0, Brut - Franchise_Cédante - Rétention_Cédante)
    
    deductions_premier_niveau = conf['valeur'] + conf['retention_cedante']
    
    brut_seuil_bas = (L / part_ccr) + deductions_premier_niveau
    brut_seuil_haut = (L_plus_M / part_ccr) + deductions_premier_niveau
    
    # Étape B : Calcul des LEV analytiques sur la courbe GPD brute
    lev_bas = limited_expected_value_gpd(brut_seuil_bas, xi, sigma, u)
    lev_haut = limited_expected_value_gpd(brut_seuil_haut, xi, sigma, u)
    
    # Étape C : Charge brute moyenne par sinistre dans cette zone
    charge_brute_sinistre_tranche = lev_haut - lev_bas
    
    # Étape D : Application du prorata de la CCR pour obtenir la charge nette par sinistre
    charge_nette_sinistre_ccr_tranche = charge_brute_sinistre_tranche * part_ccr
    
    # Étape E : Multiplier par la fréquence annuelle moyenne (loi de Poisson) pour obtenir la Prime Pure
    prime_pure_annuelle = charge_nette_sinistre_ccr_tranche * lam
    
    # Calcul du Rate on Line (ROL) théorique = Prime Pure / Portée de la tranche
    rol_theorique = (prime_pure_annuelle / M) * 100
    
    print(f"{t['nom']} :")
    print(f"  -> Espérance de charge par sinistre (CCR) : {charge_nette_sinistre_ccr_tranche:,.2f} DA")
    print(f"  -> PRIME PURE ANNUELLE THÉORIQUE          : {prime_pure_annuelle:,.2f} DA")
    print(f"  -> Rate on Line (ROL) Théorique           : {rol_theorique:.4f} %")
    print("-" * 75)

Calcul de la VaR

In [ ]:
# --- 1. PARAMÈTRES DE L'ÉTUDE DE LA SINISTRALITÉ DE LA CCR ---
xi = 0.4003      
sigma = 1_748_298_084.89   
u = 600_000_000           
lam = 4.6                
n_annees = 100000
part_ccr_dans_sinistre = 0.60  # La CCR détient 60% du sinistre net de franchise cédante

# --- PARAMÈTRES DU PROGRAMME MULTI-TRANCHES ---
priorite_ccr = 1_000_000_000  # Franchise globale conservée par la CCR (Priorité initiale)

# --- 2. GÉNÉRATION DE L'ÉCHANTILLON DE SÉVÉRITÉ (GPD) ---
np.random.seed(42)
echantillons_exces = genpareto.rvs(xi, loc=0, scale=sigma, size=10000)
echantillons_nouveau = echantillons_exces + u


# --- 3. SIMULATION DE MONTE CARLO : RECONSTITUTION DE LA CHARGE RÉELLE CCR ---
def simuler_charge_globale_ccr(echantillons_gpd, lam, n_sim, part_ccr):
    charges_annuelles_ccr = []
    noms_assures = list(dict_assures.keys())
    poids = [dict_assures[a]['proportion'] for a in noms_assures]

    for _ in range(n_sim):
        n_sin = poisson.rvs(lam)
        charge_annee_ccr = 0
        
        if n_sin > 0:
            sinistres_bruts = np.random.choice(echantillons_gpd, size=n_sin)
            assures_concernes = np.random.choice(noms_assures, size=n_sin, p=poids)
            
            for i in range(n_sin):
                s_brut = sinistres_bruts[i]
                assure = assures_concernes[i]
                conf = dict_assures[assure]
                
                if conf['type_franchise'] == 'proportion':
                    franchise = max(conf['min_f'], conf['valeur'] * s_brut)
                else:
                    franchise = conf['valeur']
                
                s_net_f_100 = max(0, s_brut - franchise)
                s_net_cedante_100 = max(0, s_net_f_100 - conf['retention_cedante'])
                
                s_part_ccr = s_net_cedante_100 * part_ccr
                charge_annee_ccr += s_part_ccr

        charges_annuelles_ccr.append(charge_annee_ccr)

    return np.array(charges_annuelles_ccr)


# --- 4. EXÉCUTION DE LA SIMULATION ---
charges_globales = simuler_charge_globale_ccr(
    echantillons_nouveau, lam, n_annees, part_ccr_dans_sinistre
)


# --- 5. CALCUL DYNAMIQUE DES QUANTILES DE LA SIMULATION ---
seuil_95 = np.percentile(charges_globales, 95.0)
seuil_98 = np.percentile(charges_globales, 98.0)
seuil_995 = np.percentile(charges_globales, 99.5)  # Solvabilité Target
seuil_999 = np.percentile(charges_globales, 99.9)  # Horizon millénaire

print(f"--- ANALYSE DES QUANTILES DE CHARGE ANNUELLE SIMULÉE ---")
print(f"Quantile 95.0% (Risque modéré)     : {seuil_95:,.2f} DA")
print(f"Quantile 98.0% (Risque fort)       : {seuil_98:,.2f} DA")
print(f"Quantile 99.5% (Risque extrême)    : {seuil_995:,.2f} DA")
print(f"Quantile 99.9% (Risque critique)   : {seuil_999:,.2f} DA")
print("-" * 65)

Architecture du NM-XL suggérée 

In [ ]:
# --- 1. PARAMÈTRES DE L'ÉTUDE DE LA SINISTRALITÉ DE LA CCR ---
xi = 0.4003      
sigma = 1_748_298_084.89   
u = 600_000_000           
lam = 4.6                
n_annees = 100000
part_ccr_dans_sinistre = 0.60  # La CCR détient 60% du sinistre net de franchise cédante

# --- PARAMÈTRES DU PROGRAMME MULTI-TRANCHES OPTIMISÉ ---
priorite_ccr = 1_000_000_000  # Nouvelle franchise globale conservée par la CCR

# Configuration de la Tranche 1 (Strate de Volatilité)
portee_t1 = 2_300_000_000      # Capacité par événement Tranche 1 (Corrigé à 3 Mds)
reinst_t1 = 3                  # Nombre de reconstitutions Tranche 1
limite_annuelle_t1 = portee_t1 * (1 + reinst_t1)

# Configuration de la Tranche 2 (Strate de Solvabilité)
portee_t2 = 4_700_000_000      # Capacité par événement Tranche 2
reinst_t2 = 2                  # Nombre de reconstitutions Tranche 2
limite_annuelle_t2 = portee_t2 * (1 + reinst_t2)

# Configuration de la Tranche 3 (Strate de Ruine / Catastrophe)
portee_t3 = 9_000_000_000     # Capacité par événement Tranche 3
reinst_t3 = 1                  # Nombre de reconstitutions Tranche 3
limite_annuelle_t3 = portee_t3 * (1 + reinst_t3)


# --- 2. GÉNÉRATION DE L'ÉCHANTILLON DE SÉVÉRITÉ (GPD) ---
np.random.seed(42)
echantillons_exces = genpareto.rvs(xi, loc=0, scale=sigma, size=10000)
echantillons_nouveau = echantillons_exces + u


# --- 3. SIMULATION DE MONTE CARLO : RECONSTITUTION DE LA CHARGE RÉELLE CCR ---
def simuler_charge_globale_ccr(echantillons_gpd, lam, n_sim, part_ccr):
    charges_annuelles_ccr = []
    
    noms_assures = list(dict_assures.keys())
    poids = [dict_assures[a]['proportion'] for a in noms_assures]

    for _ in range(n_sim):
        n_sin = poisson.rvs(lam)
        charge_annee_ccr = 0
        
        if n_sin > 0:
            sinistres_bruts = np.random.choice(echantillons_gpd, size=n_sin)
            assures_concernes = np.random.choice(noms_assures, size=n_sin, p=poids)
            
            for i in range(n_sin):
                s_brut = sinistres_bruts[i]
                assure = assures_concernes[i]
                conf = dict_assures[assure]
                
                # --- A. DÉDUCTIONS SUR LE BRUT (100%) ---
                if conf['type_franchise'] == 'proportion':
                    franchise = max(conf['min_f'], conf['valeur'] * s_brut)
                else:
                    franchise = conf['valeur']
                
                s_net_f_100 = max(0, s_brut - franchise)
                s_net_cedante_100 = max(0, s_net_f_100 - conf['retention_cedante'])
                
                # --- B. APPLICATION DE LA PART NETTE CCR (60%) ---
                s_part_ccr = s_net_cedante_100 * part_ccr
                charge_annee_ccr += s_part_ccr

        charges_annuelles_ccr.append(charge_annee_ccr)

    return np.array(charges_annuelles_ccr)


# --- 4. EXÉCUTION DE LA SIMULATION ET CASCADE DE PROTECTION ---
# Étape A : Génération des charges réelles annuelles cumulées de la CCR
charges_globales = simuler_charge_globale_ccr(
    echantillons_nouveau, lam, n_annees, part_ccr_dans_sinistre
)

charges_t1 = []
charges_t2 = []
charges_t3 = []
pertes_non_couvertes = [] 

# Étape B : Passage des charges réelles de la CCR dans la cascade des tranches
for charge_annuelle_ccr in charges_globales:
    
    # Priorité globale conservée par la CCR
    excédent_global = max(0, charge_annuelle_ccr - priorite_ccr)
    
    # Absorption par la Tranche 1
    paye_t1 = min(excédent_global, limite_annuelle_t1)
    charges_t1.append(paye_t1)
    
    # Le surplus monte en Tranche 2
    surplus_vers_t2 = max(0, excédent_global - paye_t1)
    paye_t2 = min(surplus_vers_t2, limite_annuelle_t2)
    charges_t2.append(paye_t2)
    
    # Le surplus résiduel monte en Tranche 3
    surplus_vers_t3 = max(0, surplus_vers_t2 - paye_t2)
    paye_t3 = min(surplus_vers_t3, limite_annuelle_t3)
    charges_t3.append(paye_t3)
    
    # Ce qui dépasse tout le programme (Risque de ruine sur fonds propres)
    reste = max(0, surplus_vers_t3 - paye_t3)
    pertes_non_couvertes.append(reste)

# Conversion
charges_t1 = np.array(charges_t1)
charges_t2 = np.array(charges_t2)
charges_t3 = np.array(charges_t3)
pertes_non_couvertes = np.array(pertes_non_couvertes)


# --- 5. STATISTIQUES ET ALIMENTATION DES RÉSULTATS DU MÉMOIRE ---
sat_t1 = (charges_t1 >= limite_annuelle_t1).mean() * 100
sat_t2 = (charges_t2 >= limite_annuelle_t2).mean() * 100
sat_t3 = (charges_t3 >= limite_annuelle_t3).mean() * 100

var_globale_995 = np.percentile(charges_globales, 99.5)
var_globale_999 = np.percentile(charges_globales, 99.9)

print(f"--- 1. ÉVALUATION DU RISQUE REEL GLOBAL DE LA CCR (AVANT RETROCESSION) ---")
print(f"VaR REELLE GLOBALE À 99.5% : {var_globale_995:,.2f} DA")
print(f"VaR REELLE GLOBALE À 99.9% : {var_globale_999:,.2f} DA")
print("-" * 65)

print(f"--- 2. DIAGNOSTIC DE LA CASCADE DE PROTECTION MULTI-TRANCHES ---")
print(f"Probabilité de saturation de la Tranche 1 : {sat_t1:.3f} %  (Limite annuelle : {limite_annuelle_t1:,.0f} DA)")
print(f"Probabilité de saturation de la Tranche 2 : {sat_t2:.3f} %  (Limite annuelle : {limite_annuelle_t2:,.0f} DA)")
print(f"Probabilité de saturation de la Tranche 3 : {sat_t3:.3f} %  (Limite annuelle : {limite_annuelle_t3:,.0f} DA)")
print("-" * 65)

print(f"--- 3. MESURE DE L'ÉTANCHÉITÉ (RISQUE NET NON COUVERT DU PORTEFEUILLE) ---")
print(f"VaR 99.5% du risque NON COUVERT (Impact direct fonds propres) : {np.percentile(pertes_non_couvertes, 99.5):,.2f} DA")
print(f"VaR 99.9% du risque NON COUVERT (Impact direct fonds propres) : {np.percentile(pertes_non_couvertes, 99.9):,.2f} DA")
print("-" * 65)

In [ ]:
# Définition des frontières financières (seuils cumulés au-dessus de 0)
seuil_bas_t1 = priorite_ccr
seuil_haut_t1 = priorite_ccr + limite_annuelle_t1

seuil_bas_t2 = seuil_haut_t1
seuil_haut_t2 = seuil_bas_t2 + limite_annuelle_t2

seuil_bas_t3 = seuil_haut_t2
seuil_haut_t3 = seuil_bas_t3 + limite_annuelle_t3

# Calcul des quantiles exacts correspondants (en % entre 0 et 100)
quantile_bas_t1 = percentileofscore(charges_globales, seuil_bas_t1)
quantile_haut_t1 = percentileofscore(charges_globales, seuil_haut_t1)

quantile_bas_t2 = percentileofscore(charges_globales, seuil_bas_t2)
quantile_haut_t2 = percentileofscore(charges_globales, seuil_haut_t2)

quantile_bas_t3 = percentileofscore(charges_globales, seuil_bas_t3)
quantile_haut_t3 = percentileofscore(charges_globales, seuil_haut_t3)

print(f"--- MAPPING DES QUANTILES PAR TRANCHE DE PROTECTION ---")
print(f"RÉTENTION NETTE CCR : Couvre de 0.00 % à {quantile_bas_t1:.3f} % des années.")
print("-" * 75)
print(f"TRANCHE 1 (12 Mds de limite annuelle) :")
print(f"  -> Protège le portefeuille entre les quantiles {quantile_bas_t1:.3f} % et {quantile_haut_t1:.3f} %")
print("-" * 75)
print(f"TRANCHE 2 (15 Mds de limite annuelle) :")
print(f"  -> Protège le portefeuille entre les quantiles {quantile_bas_t2:.3f} % et {quantile_haut_t2:.3f} %")
print("-" * 75)
print(f"TRANCHE 3 (30 Mds de limite annuelle) :")
print(f"  -> Protège le portefeuille entre les quantiles {quantile_bas_t3:.3f} % et {quantile_haut_t3:.3f} %")
print("-" * 75)
print(f"ZONE DE DEPASSEMENT (Risque de ruine ultime) :")
print(f"  -> Tout sinistre au-delà du quantile {quantile_haut_t3:.3f} % impacte les fonds propres.")
print("-" * 75)

Critère de nullité de la VaR risiduelle 

In [ ]:
# --- 6. RECHERCHE DU SEUIL DE CONFIANCE OÙ LA VaR RÉSIDUELLE DEVIENT NULLE ---

# On cherche la proportion d'années où les pertes non couvertes sont égales à 0
probabilite_zero_perte = (pertes_non_couvertes == 0).mean()

# Le seuil de confiance exact (le quantile) où la VaR atteint 0 correspond à cette proportion
seuil_exact_pour_zero = probabilite_zero_perte * 100

# Calcul de la période de retour correspondante à la première perte non nulle
# Si le programme couvre jusqu'au seuil X%, la période de retour est 1 / (1 - X)
if probabilite_zero_perte < 1.0:
    periode_retour_zero = 1 / (1 - probabilite_zero_perte)
    affichage_periode = f"{periode_retour_zero:,.1f} ans"
else:
    affichage_periode = "Infinie (Le programme n'est jamais dépassé)"

print(f"--- OPTIMISATION DU SEUIL DE SÉCURITÉ ABSOLUE ---")
print(f"La VaR résiduelle atteint EXACTEMENT 0.00 DA au seuil de : {seuil_exact_pour_zero:.4f} %")
print(f"Période de retour de la première perte pour la CCR     : {affichage_periode}")
print("-" * 65)

Définition du montant de la rétention equivalent au quantile de 10% 

In [ ]:
# --- 8. CALCUL DU MONTANT CORRESPONDANT À LA VaR 10% ---

# VaR 10% sur la charge globale brute de la CCR
var_globale_10 = np.percentile(charges_globales, 97.5)

# VaR 10% sur le risque net non couvert (après rétrocession)
var_non_couverte_10 = np.percentile(pertes_non_couvertes, 97.5)

print(f"--- ANALYSE DE LA VaR AU SEUIL 10% ---")
print(f"Montant de la VaR Globale (Brute CCR) à 10% : {var_globale_10:,.2f} DA")
print(f"Montant de la VaR Résiduelle (Nette) à 10%   : {var_non_couverte_10:,.2f} DA")
print(f"-> Signification : Dans 90% des cas, la charge annuelle sera supérieure à ces montants.")
print("-" * 65)